<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/horizyn_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 强制回到 Colab 根目录，防止多次运行导致的路径嵌套
%cd /content

# （可选清理）为了防止之前克隆的代码损坏，先删除旧文件夹，重新拉取干净的代码
!rm -rf horizyn
!git clone https://github.com/dayhofflabs/horizyn.git

# 2. 绝对路径进入项目根目录
%cd /content/horizyn

# 3. 安装包管理器并同步依赖（这一步会在内部自动生成并配置.venv）
!pip install uv
!uv sync

# 4. 使用 uv run 执行下载数据脚本（uv run 会自动寻找并使用正确的虚拟环境）
!uv run python scripts/download_data.py

# Diagnose missing file: Check contents of the data/sota directory
!ls -l data/sota

# 5. 启动模型训练
!uv run python train.py --config configs/sota.yaml

/content
Cloning into 'horizyn'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 102 (delta 12), reused 18 (delta 4), pack-reused 19 (from 1)
Receiving objects: 100% (102/102), 369.74 KiB | 19.46 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/horizyn
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 96 packages in 1ms
Prepared 1 package in 416ms
Installed 88 packages in 222ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + attrs==25.4.0
 + black==25.9.0
 + blosc2==3.11.0
 + certifi==2025.10.5
 + cfgv==3.4.0
 + charset-normalizer==3.4.4
 + click==8.3.0
 + coverage==7.11.0
 + distlib==0.4.0
 + drfp==0.3.7
 + et-xmlfile==2.0.0
 + filelock==3.20.0
 + flake8==7.3.0
 + frozenlist==1.8.0
 + fsspec==2025.10.0
 + greenlet==3.2.4
 + h5py==3.15.1
 + horizyn==1.0.0 (from file:///content/horizyn)
 + identify==2.

In [ ]:
# 启动训练
!uv run python train.py --config configs/sota.yaml

Loading config from: configs/sota.yaml

HORIZYN TRAINING CONFIGURATION
Seed: 42
Max Epochs: 100
Train Batch Size: 131072
Retrieval Batch Size: 128
Learning Rate: 0.0001
Weight Decay: 0.01
Model: DualContrastiveModel
Query Encoder: [2048, 4096, 4096, 512]
Target Encoder: [1024, 4096, 4096, 512]
Embedding Dim: 512
Loss: FullBatchMLNCELoss (beta=10.0)
Log Dir: logs
Checkpoint Dir: /content/drive/MyDrive/Horizyn_Checkpoints

Seed set to 42
Set random seed to: 42

Initializing data module...
Data module initialized.

Initializing model...
Total parameters: 50,349,057
Trainable parameters: 50,349,056

Setting up Lightning Trainer...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Trainer configured for 100 epochs

STARTING TRAINING

You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more 

In [ ]:
import yaml
import os

# 1. 定义配置文件路径和你的 Google Drive 目标路径
config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 2. 确保 Google Drive 中的目标文件夹存在（如果不存在则自动创建）
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 3. 读取当前的 yaml 配置文件
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

# 4. 强制修改 logging 模块下的 checkpoint_dir 路径
if 'logging' in config:
    config['logging']['checkpoint_dir'] = drive_checkpoint_path
else:
    # 兼容性处理：如果原文件碰巧没有 logging 项，则主动创建
    config['logging'] = {'checkpoint_dir': drive_checkpoint_path}

# 5. 将修改后的配置重新写回文件，保持格式整洁
with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print(f"✅ 检查点保存路径已成功强制修改为: {drive_checkpoint_path}")

# 6. 使用 Linux 命令打印该行，验证是否写入硬盘
!echo "当前文件中的路径配置为："
!cat configs/sota.yaml | grep "checkpoint_dir"

✅ 检查点保存路径已成功强制修改为: /content/drive/MyDrive/Horizyn_Checkpoints
当前文件中的路径配置为：
  checkpoint_dir: /content/drive/MyDrive/Horizyn_Checkpoints


In [ ]:
import yaml
import os
import glob
import re

config_path = 'configs/sota.yaml'
drive_checkpoint_path = '/content/drive/MyDrive/Horizyn_Checkpoints'

# 1. 创建云盘保存路径（防断连的终极保障）
os.makedirs(drive_checkpoint_path, exist_ok=True)

# 2. 彻底且正确地覆写 YAML 核心配置
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

# 修复拼写错误，强制注入性能加速参数
if 'data' not in config: config['data'] = {}
config['data']['train_batch_size'] = 131072
if 'in_memory' in config['data']:
    del config['data']['in_memory'] # 自动清理错误拼写
config['data']['pin_memory'] = True

# 强制改为：每一轮结束都必须保存一次！
if 'logging' not in config: config['logging'] = {}
config['logging']['checkpoint_dir'] = drive_checkpoint_path
config['logging']['save_every_n_epochs'] = 1

with open(config_path, 'w', encoding='utf-8') as file:
    yaml.dump(config, file, default_flow_style=False, sort_keys=False)

print("✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。")

# 3. 高级智能续训补丁：底层拦截并修改 Trainer.fit
ckpt_list = glob.glob(os.path.join(drive_checkpoint_path, '*.ckpt'))
latest_ckpt = max(ckpt_list, key=os.path.getctime) if ckpt_list else None

with open('train.py', 'r', encoding='utf-8') as f:
    content = f.read()

# 清理旧的拦截补丁（防止多次运行重复注入）
content = re.sub(r'# --- AUTO INJECTED RESUME PATCH ---.*?# ----------------------------------\n', '', content, flags=re.DOTALL)

if latest_ckpt:
    print(f"✅ 发现最新历史检查点：{os.path.basename(latest_ckpt)}")
    print("🚀 本次运行将无缝继承优化器与学习率，从断点处直接继续！")

    # 猴子补丁：直接在代码执行内存中强行挂载 ckpt_path，绝对不会引发语法错误
    patch_code = f"""# --- AUTO INJECTED RESUME PATCH ---
import pytorch_lightning as pl
_original_fit = pl.Trainer.fit
def _patched_fit(self, *args, **kwargs):
    kwargs['ckpt_path'] = r"{latest_ckpt}"
    print(f"\\n 成功挂载历史权重，从检查点恢复训练...\\n")
    return _original_fit(self, *args, **kwargs)
pl.Trainer.fit = _patched_fit
# ----------------------------------\n"""
    content = patch_code + content
else:
    print("⚠️ 云盘中暂无检查点。本次将从 Epoch 0 全新开始，跑完第 1 轮后就会自动产生备份！")

with open('train.py', 'w', encoding='utf-8') as f:
    f.write(content)

✅ 配置文件已强制刷新：开启极大Batch、锁页内存、每 1 轮自动备份。
⚠️ 云盘中暂无检查点。本次将从 Epoch 0 全新开始，跑完第 1 轮后就会自动产生备份！


In [ ]:
import yaml

config_path = 'configs/sota.yaml'

# 读取当前的配置文件
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

# 强制修改参数（确保是在 data 模块下）
config['data']['train_batch_size'] = 131072
config['data']['pin_memory'] = True

# 将修改后的配置重新写回文件
with open(config_path, 'w') as file:
    yaml.dump(config, file)

print("配置已成功更新并强制写入硬盘！")

# 打印出来验证一下
!cat configs/sota.yaml | grep -E "train_batch_size|pin_memory"

配置已成功更新并强制写入硬盘！
  pin_memory: true
  train_batch_size: 131072


In [ ]:
import IPython
from google.colab import output

display(IPython.display.Javascript('''
function ClickConnect(){
    // 自动点击连接按钮
    let btn = document.querySelector("colab-connect-button");
    if (btn!= null){
        console.log("正在保持 Colab 活跃...");
        btn.click();
    }
    // 自动处理可能弹出的确认对话框
    let ok_btn = document.getElementById('ok');
    if (ok_btn!= null){
        console.log("处理弹窗...");
        ok_btn.click();
    }
}
// 每 60 秒自动执行一次
setInterval(ClickConnect, 60000);
'''))
print("防断连脚本已在后台启动！")

<IPython.core.display.Javascript object>

防断连脚本已在后台启动！


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. 确保在项目根目录
%cd /content/horizyn

# 2. 强制创建 sota.yaml 期望的文件夹路径
!mkdir -p data/sota

# 3. 如果文件错位下载到了外面，将它们移动到正确的文件夹中
!mv prots_t5.h5 data/sota/ 2>/dev/null || true
!mv *.csv data/sota/ 2>/dev/null || true
!mv data/*.csv data/sota/ 2>/dev/null || true
!mv data/*.h5 data/sota/ 2>/dev/null || true

# 4. 安全起见，再次执行官方下载脚本（如果文件已存在且完整，脚本通常会跳过下载）
!uv run python scripts/download_data.py

# 5. 再次启动模型训练
!uv run python train.py --config configs/sota.yaml

SyntaxError: invalid syntax (2978541721.py, line 10)